# Q1 — Text Classification (IMDb)

**Canonical path**: Matched 4K train / 2K test comparison across TF-IDF LR, TF-IDF SVM, BiLSTM, DistilBERT.

**Runtime**: T4 GPU required for BiLSTM and DistilBERT cells.

> This notebook reproduces the report-facing Q1 comparison artifacts.
> For full 25K IMDb runs, see the **Exploratory** section at the bottom.

## 1. Environment Setup

In [ ]:
import os, sys, json, glob, shutil

# --- Google Drive mount (Colab only) ---
try:
    from google.colab import drive
    drive.mount('/content/drive')
    ON_COLAB = True
except ImportError:
    ON_COLAB = False
    print('Not running on Colab — skipping Drive mount.')

DRIVE_OUTPUT = '/content/drive/MyDrive/467-takehome-outputs/q1' if ON_COLAB else None
if DRIVE_OUTPUT:
    os.makedirs(DRIVE_OUTPUT, exist_ok=True)

In [ ]:
REPO_DIR = '/content/467-takehome' if ON_COLAB else os.path.abspath('..')
if ON_COLAB:
    if not os.path.exists(REPO_DIR):
        !git clone https://github.com/zubeyralmaho/467-takehome.git {REPO_DIR}
    else:
        !cd {REPO_DIR} && git pull --ff-only
os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import torch
print(f'PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  GPU: {torch.cuda.get_device_name(0)}  |  VRAM: {torch.cuda.get_device_properties(0).total_mem/1e9:.1f} GB')

In [ ]:
def save_to_drive(run_dir, tag=''):
    """Copy a run directory to Google Drive (no-op outside Colab)."""
    if not DRIVE_OUTPUT:
        return
    name = os.path.basename(run_dir) + (f'_{tag}' if tag else '')
    dest = os.path.join(DRIVE_OUTPUT, name)
    shutil.copytree(run_dir, dest, dirs_exist_ok=True)
    print(f'Saved to Drive: {dest}')

def gpu_cleanup():
    """Free GPU memory between model runs."""
    import gc
    torch.cuda.empty_cache()
    gc.collect()
    if torch.cuda.is_available():
        print(f'GPU memory freed. Allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB')

## 2. Canonical Run — Matched 4K/2K Comparison

Reproduces the report-facing comparison: 4 000 train / 2 000 test, all 4 models on the same budget.

### 2a. TF-IDF LR + SVM (CPU)

In [ ]:
!python -m src.q1_classification.main \
    --config configs/q1.yaml \
    --final-eval \
    --override \
        dataset.limit_train_samples=4000 \
        dataset.limit_test_samples=2000 \
        models.tfidf_lr.enabled=true \
        models.tfidf_svm.enabled=true \
        models.bilstm.enabled=false \
        models.distilbert.enabled=false

In [ ]:
tfidf_run = sorted(glob.glob('outputs/q1/run_*'))[-1]
save_to_drive(tfidf_run, 'tfidf')
print(f'TF-IDF run: {tfidf_run}')

### 2b. BiLSTM (GPU)

In [ ]:
!python -m src.q1_classification.main \
    --config configs/q1.yaml \
    --final-eval \
    --override \
        dataset.limit_train_samples=4000 \
        dataset.limit_test_samples=2000 \
        models.tfidf_lr.enabled=false \
        models.tfidf_svm.enabled=false \
        models.bilstm.enabled=true \
        models.distilbert.enabled=false \
        models.bilstm.max_epochs=10 \
        models.bilstm.batch_size=64

In [ ]:
bilstm_run = sorted(glob.glob('outputs/q1/run_*'))[-1]
save_to_drive(bilstm_run, 'bilstm')
gpu_cleanup()

### 2c. DistilBERT (GPU)

In [ ]:
!python -m src.q1_classification.main \
    --config configs/q1.yaml \
    --final-eval \
    --override \
        dataset.limit_train_samples=4000 \
        dataset.limit_test_samples=2000 \
        models.tfidf_lr.enabled=false \
        models.tfidf_svm.enabled=false \
        models.bilstm.enabled=false \
        models.distilbert.enabled=true \
        models.distilbert.max_epochs=3 \
        models.distilbert.batch_size=16

In [ ]:
distilbert_run = sorted(glob.glob('outputs/q1/run_*'))[-1]
save_to_drive(distilbert_run, 'distilbert')
gpu_cleanup()

## 3. Generate Report Comparison Artifact

Runs the Q1 report summary script to produce the comparison CSV and LaTeX table from the runs above.

In [ ]:
# Update these paths to the runs produced above
print('Canonical Q1 runs:')
print(f'  TF-IDF:     {tfidf_run}')
print(f'  BiLSTM:     {bilstm_run}')
print(f'  DistilBERT: {distilbert_run}')
print()
print('To generate the report comparison artifact, run:')
print(f'  python scripts/q1_report_summary.py')  # uses canonical paths defined in the script

## 4. Results Summary

In [ ]:
print('=== Q1 Canonical Results ===')
for label, run_dir in [('TF-IDF', tfidf_run), ('BiLSTM', bilstm_run), ('DistilBERT', distilbert_run)]:
    metrics_file = os.path.join(run_dir, 'metrics.json')
    if os.path.exists(metrics_file):
        with open(metrics_file) as f:
            m = json.load(f)
        print(f'\n--- {label} ({os.path.basename(run_dir)}) ---')
        print(json.dumps(m, indent=2, default=str)[:1200])
    else:
        print(f'\n--- {label}: metrics.json not found ---')

---
## (Exploratory) Full 25K IMDb Runs

These runs use the full IMDb dataset and are **not** part of the canonical report comparison.
Uncomment the cells below if needed.

In [ ]:
# # --- Full 25K TF-IDF ---
# !python -m src.q1_classification.main \
#     --config configs/q1.yaml \
#     --final-eval \
#     --override \
#         models.tfidf_lr.enabled=true \
#         models.tfidf_svm.enabled=true \
#         models.bilstm.enabled=false \
#         models.distilbert.enabled=false

# # --- Full 25K BiLSTM ---
# !python -m src.q1_classification.main \
#     --config configs/q1.yaml \
#     --final-eval \
#     --override \
#         models.bilstm.enabled=true \
#         models.tfidf_lr.enabled=false \
#         models.tfidf_svm.enabled=false \
#         models.distilbert.enabled=false \
#         models.bilstm.max_epochs=10

# # --- Full 25K DistilBERT ---
# !python -m src.q1_classification.main \
#     --config configs/q1.yaml \
#     --final-eval \
#     --override \
#         models.distilbert.enabled=true \
#         models.tfidf_lr.enabled=false \
#         models.tfidf_svm.enabled=false \
#         models.bilstm.enabled=false \
#         models.distilbert.max_epochs=3

# Q1 — Text Classification (IMDb)
**Models:** TF-IDF LR/SVM (CPU) → BiLSTM (GPU) → DistilBERT (GPU)

**Dataset:** Full 25K IMDb training set

**Runtime:** T4 GPU required

## 1. Environment Setup

In [ ]:
import os, subprocess, sys

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Output directory on Drive
DRIVE_OUTPUT = '/content/drive/MyDrive/467-takehome-outputs/q1'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
print('Drive mounted, output dir ready.')

In [ ]:
# Clone repo
REPO_DIR = '/content/467-takehome'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/zubeyralmaho/467-takehome.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

In [ ]:
# Install dependencies
!pip install -q -r requirements.txt

In [ ]:
# Verify GPU
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

## 2. Run TF-IDF Models (CPU) — LR + SVM

In [ ]:
!python -m src.q1_classification.main \
    --config configs/q1.yaml \
    --final-eval \
    --override \
        models.tfidf_lr.enabled=true \
        models.tfidf_svm.enabled=true \
        models.bilstm.enabled=false \
        models.distilbert.enabled=false

In [ ]:
# Save TF-IDF results to Drive
import glob, shutil
latest_run = sorted(glob.glob('outputs/q1/run_*'))[-1]
dest = os.path.join(DRIVE_OUTPUT, os.path.basename(latest_run) + '_tfidf')
shutil.copytree(latest_run, dest, dirs_exist_ok=True)
print(f'TF-IDF results saved to Drive: {dest}')

## 3. Run BiLSTM (GPU)

In [ ]:
!python -m src.q1_classification.main \
    --config configs/q1.yaml \
    --final-eval \
    --override \
        models.tfidf_lr.enabled=false \
        models.tfidf_svm.enabled=false \
        models.bilstm.enabled=true \
        models.distilbert.enabled=false \
        models.bilstm.max_epochs=10 \
        models.bilstm.batch_size=64

In [ ]:
# Save BiLSTM results to Drive
latest_run = sorted(glob.glob('outputs/q1/run_*'))[-1]
dest = os.path.join(DRIVE_OUTPUT, os.path.basename(latest_run) + '_bilstm')
shutil.copytree(latest_run, dest, dirs_exist_ok=True)
print(f'BiLSTM results saved to Drive: {dest}')

In [ ]:
# Free GPU memory before next model
import torch, gc
torch.cuda.empty_cache()
gc.collect()
print(f"GPU memory freed. Allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")

## 4. Run DistilBERT (GPU)

In [ ]:
!python -m src.q1_classification.main \
    --config configs/q1.yaml \
    --final-eval \
    --override \
        models.tfidf_lr.enabled=false \
        models.tfidf_svm.enabled=false \
        models.bilstm.enabled=false \
        models.distilbert.enabled=true \
        models.distilbert.max_epochs=3 \
        models.distilbert.batch_size=16

In [ ]:
# Save DistilBERT results to Drive
latest_run = sorted(glob.glob('outputs/q1/run_*'))[-1]
dest = os.path.join(DRIVE_OUTPUT, os.path.basename(latest_run) + '_distilbert')
shutil.copytree(latest_run, dest, dirs_exist_ok=True)
print(f'DistilBERT results saved to Drive: {dest}')

## 5. Results Summary

In [ ]:
import json

print('=== Q1 Results Summary ===')
for run_dir in sorted(glob.glob('outputs/q1/run_*')):
    metrics_file = os.path.join(run_dir, 'metrics.json')
    if os.path.exists(metrics_file):
        with open(metrics_file) as f:
            metrics = json.load(f)
        print(f'\n--- {os.path.basename(run_dir)} ---')
        print(json.dumps(metrics, indent=2, default=str)[:1500])